# CIC-IDS2017: preprocessing and feature selection

This notebook contains the complete code for the first two project stages. It does not import project functions from `src/`; every operation is visible here.

Pipeline: **CSV files → cleaning → train/test split → scaling → feature selection**

## 1. Imports and settings

In [ ]:
from pathlib import Path
import json
import math

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 100)
RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

CSV_FOLDER = PROJECT_ROOT / "dataset/CSVs/MachineLearningCSV"
PCAP_FOLDER = PROJECT_ROOT / "dataset/PCAPs"
PROCESSED_FOLDER = PROJECT_ROOT / "dataset/processed"
FEATURE_FOLDER = PROJECT_ROOT / "artifacts/feature_selection"

# Set this to True when you want to recreate train.parquet and test.parquet.
RUN_PREPROCESSING = False

# Set this to True when you want to recalculate feature scores.
RUN_FEATURE_SELECTION = False

## 2. Input files

The machine-learning CSV files already contain features extracted by CICFlowMeter. PCAP files are listed separately for future raw-flow extraction.

In [ ]:
csv_files = sorted(CSV_FOLDER.glob("*.csv"))
pcap_files = sorted(PCAP_FOLDER.rglob("*.pcap")) if PCAP_FOLDER.exists() else []

print(f"CSV files:  {len(csv_files)}")
print(f"PCAP files: {len(pcap_files)}")
pd.DataFrame({
    "CSV file": [file.name for file in csv_files],
    "Size (MB)": [round(file.stat().st_size / 1024**2, 2) for file in csv_files],
})

## 3. Load all CSV files

In [ ]:
def load_csv_files(files):
    dataframes = []
    for file in tqdm(files, desc="Reading CSV files", unit="file"):
        dataframes.append(pd.read_csv(file, low_memory=False))
    return pd.concat(dataframes, ignore_index=True)

In [ ]:
if RUN_PREPROCESSING:
    raw_data = load_csv_files(csv_files)
    print(f"Rows: {len(raw_data):,}")
    print(f"Columns: {len(raw_data.columns)}")
    display(raw_data.head())
else:
    print("Skipped. Set RUN_PREPROCESSING = True to load all raw CSV files.")

## 4. Clean the data

Cleaning removes whitespace from headers, fixes labels, converts features to numbers, removes invalid and duplicate rows, removes one repeated column, and creates a binary target.

In [ ]:
def clean_data(data):
    data = data.copy()
    input_rows = len(data)

    data.columns = data.columns.str.strip()
    data["Label"] = (
        data["Label"].astype("string").str.strip()
        .str.replace("�", "-", regex=False)
        .str.replace(r"\s*-\s*", " - ", regex=True)
    )

    feature_columns = [column for column in data.columns if column != "Label"]
    for column in tqdm(feature_columns, desc="Converting features"):
        data[column] = pd.to_numeric(data[column], errors="coerce")

    data.replace([np.inf, -np.inf], np.nan, inplace=True)
    before_invalid = len(data)
    data.dropna(inplace=True)
    invalid_rows = before_invalid - len(data)

    repeated_column = "Fwd Header Length.1"
    if repeated_column in data.columns:
        if not data[repeated_column].equals(data["Fwd Header Length"]):
            raise ValueError(f"{repeated_column} is not an exact copy")
        data.drop(columns=repeated_column, inplace=True)

    before_duplicates = len(data)
    data.drop_duplicates(inplace=True)
    duplicate_rows = before_duplicates - len(data)

    data.rename(columns={"Label": "attack_label"}, inplace=True)
    data["is_attack"] = (data["attack_label"] != "BENIGN").astype("int8")

    feature_columns = [
        column for column in data.columns
        if column not in {"attack_label", "is_attack"}
    ]
    data[feature_columns] = data[feature_columns].astype("float32")

    report = {
        "input": input_rows,
        "invalid": invalid_rows,
        "duplicates": duplicate_rows,
        "retained": len(data),
    }
    return data.reset_index(drop=True), report

In [ ]:
if RUN_PREPROCESSING:
    clean_dataset, preprocessing_report = clean_data(raw_data)
    display(pd.Series(preprocessing_report).to_frame("rows"))
    display(clean_dataset["attack_label"].value_counts().to_frame("rows"))

## 5. Split and standardize

The split is stratified so train and test have the same benign/attack ratio. The scaler is fitted only on training data to prevent leakage.

In [ ]:
def split_and_scale(data):
    feature_columns = [
        column for column in data.columns
        if column not in {"attack_label", "is_attack"}
    ]

    split = train_test_split(
        data[feature_columns],
        data["attack_label"],
        data["is_attack"],
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=data["is_attack"],
    )
    x_train, x_test, names_train, names_test, y_train, y_test = split

    scaler = StandardScaler()
    x_train = pd.DataFrame(
        scaler.fit_transform(x_train).astype("float32"), columns=feature_columns
    )
    x_test = pd.DataFrame(
        scaler.transform(x_test).astype("float32"), columns=feature_columns
    )

    train = x_train.assign(
        attack_label=names_train.reset_index(drop=True),
        is_attack=y_train.reset_index(drop=True),
    )
    test = x_test.assign(
        attack_label=names_test.reset_index(drop=True),
        is_attack=y_test.reset_index(drop=True),
    )
    return train, test, scaler, feature_columns

In [ ]:
if RUN_PREPROCESSING:
    train, test, scaler, feature_columns = split_and_scale(clean_dataset)

    PROCESSED_FOLDER.mkdir(parents=True, exist_ok=True)
    train.to_parquet(PROCESSED_FOLDER / "train.parquet", index=False)
    test.to_parquet(PROCESSED_FOLDER / "test.parquet", index=False)
    joblib.dump(scaler, PROCESSED_FOLDER / "standard_scaler.joblib")

    preprocessing_report["train"] = len(train)
    preprocessing_report["test"] = len(test)
    preprocessing_report["features"] = len(feature_columns)
    (PROCESSED_FOLDER / "preprocessing_report.json").write_text(
        json.dumps(preprocessing_report, indent=2), encoding="utf-8"
    )
    print("Saved train.parquet and test.parquet")

## 6. Inspect the prepared data

In [ ]:
report_path = PROCESSED_FOLDER / "preprocessing_report.json"
if report_path.exists():
    saved_report = json.loads(report_path.read_text(encoding="utf-8"))
    display(pd.Series(saved_report).to_frame("value"))

train_file = pq.ParquetFile(PROCESSED_FOLDER / "train.parquet")
test_file = pq.ParquetFile(PROCESSED_FOLDER / "test.parquet")
pd.DataFrame([
    {"dataset": "train", "rows": train_file.metadata.num_rows, "columns": train_file.metadata.num_columns},
    {"dataset": "test", "rows": test_file.metadata.num_rows, "columns": test_file.metadata.num_columns},
])

## 7. Sample training data for feature selection

Feature selection uses 200,000 randomly selected training rows. The test set is never used.

In [ ]:
def load_training_sample(path, sample_size=200_000):
    parquet = pq.ParquetFile(path)
    total_rows = parquet.metadata.num_rows
    sample_size = min(sample_size, total_rows)

    random = np.random.default_rng(RANDOM_STATE)
    selected_rows = np.sort(random.choice(total_rows, sample_size, replace=False))

    samples = []
    start = 0
    batch_size = 100_000
    batches = parquet.iter_batches(batch_size=batch_size)
    total_batches = math.ceil(total_rows / batch_size)

    for batch in tqdm(batches, total=total_batches, desc="Sampling training data"):
        end = start + len(batch)
        wanted = selected_rows[(selected_rows >= start) & (selected_rows < end)] - start
        if len(wanted):
            samples.append(batch.take(pa.array(wanted)).to_pandas())
        start = end

    return pd.concat(samples, ignore_index=True)

In [ ]:
if RUN_FEATURE_SELECTION:
    training_sample = load_training_sample(PROCESSED_FOLDER / "train.parquet")
    print(f"Sample rows: {len(training_sample):,}")
    display(training_sample.head())

## 8. Calculate feature scores

Each feature receives three scores: absolute correlation, mutual information, and Random Forest importance. Scores are normalized to 0–1 and averaged.

In [ ]:
def normalize(scores):
    if scores.max() == scores.min():
        return pd.Series(0.0, index=scores.index)
    return (scores - scores.min()) / (scores.max() - scores.min())


def calculate_feature_scores(data):
    feature_columns = [
        column for column in data.columns
        if column not in {"attack_label", "is_attack"}
    ]
    constant_columns = [
        column for column in feature_columns
        if data[column].nunique(dropna=False) <= 1
    ]
    feature_columns = [column for column in feature_columns if column not in constant_columns]

    features = data[feature_columns]
    labels = data["is_attack"]

    correlation = features.corrwith(labels).abs()
    mutual_information = pd.Series(
        mutual_info_classif(features, labels, random_state=RANDOM_STATE),
        index=feature_columns,
    )

    forest = RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        class_weight="balanced_subsample",
        n_jobs=1,
        random_state=RANDOM_STATE,
    )
    forest.fit(features, labels)
    forest_importance = pd.Series(forest.feature_importances_, index=feature_columns)

    scores = pd.DataFrame({
        "correlation": normalize(correlation),
        "mutual_information": normalize(mutual_information),
        "random_forest": normalize(forest_importance),
    })
    scores["combined_score"] = scores.mean(axis=1)
    scores.sort_values("combined_score", ascending=False, inplace=True)
    scores.insert(0, "rank", range(1, len(scores) + 1))
    scores.index.name = "feature"
    return scores, constant_columns

In [ ]:
if RUN_FEATURE_SELECTION:
    feature_scores, constant_features = calculate_feature_scores(training_sample)
    selected_features = feature_scores.head(20).index.tolist()
    display(feature_scores.head(20))

## 9. Save and visualize selected features

In [ ]:
if RUN_FEATURE_SELECTION:
    FEATURE_FOLDER.mkdir(parents=True, exist_ok=True)
    feature_scores.to_csv(FEATURE_FOLDER / "feature_scores.csv")

    selection_report = {
        "sample_rows": len(training_sample),
        "candidate_features": len(feature_scores),
        "constant_features": constant_features,
        "selected_feature_count": len(selected_features),
        "selected_features": selected_features,
    }
    (FEATURE_FOLDER / "selected_features.json").write_text(
        json.dumps(selection_report, indent=2), encoding="utf-8"
    )

    top = feature_scores.head(20).sort_values("combined_score")
    top["combined_score"].plot(kind="barh", figsize=(10, 8), color="steelblue")
    plt.title("Top CIC-IDS2017 features")
    plt.xlabel("Combined importance score")
    plt.tight_layout()
    plt.savefig(FEATURE_FOLDER / "top_features.png", dpi=150)
    plt.show()

## 10. View saved feature-selection results

This cell works even when `RUN_FEATURE_SELECTION` is `False`, provided feature selection has already been run once.

In [ ]:
score_path = FEATURE_FOLDER / "feature_scores.csv"
if score_path.exists():
    saved_scores = pd.read_csv(score_path).set_index("feature")
    display(saved_scores.head(20))

    saved_scores.head(20).sort_values("combined_score")["combined_score"].plot(
        kind="barh", figsize=(10, 8), color="steelblue"
    )
    plt.title("Selected features")
    plt.xlabel("Combined importance score")
    plt.tight_layout()
    plt.show()
else:
    print("Set RUN_FEATURE_SELECTION = True and run the feature-selection cells.")

## Summary

- Raw inputs are the eight machine-learning CSV files.
- Invalid and duplicate flows are removed.
- `BENIGN` is encoded as 0 and every attack as 1.
- Scaling is learned from training data only.
- Feature selection uses training data only.
- The top 20 features are chosen from correlation, mutual information, and Random Forest scores.